<a href="https://colab.research.google.com/github/fdac25/MP2/blob/main/Vis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import pandas as pd
import matplotlib.pyplot as plt



In [30]:
%pip install python-woc

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [31]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
import time

woc = WocMapsRemote(base_url="https://worldofcode.org/api/")

projects = [
    "evenharrison_finalfit",
    "lucidrains_lightweight-gan",
    "walaj_seqlib",
    "3dct_open_ia",
    "cloiupyter_cloiupyter",
    "billennium_regae",
    "cbroeckl_ramclustr",
    "jsgoecke_tesla",
    "robotology_ycm",
    "hasadna_anyway-newsflash-infographics"
]

projects

['evenharrison_finalfit',
 'lucidrains_lightweight-gan',
 'walaj_seqlib',
 '3dct_open_ia',
 'cloiupyter_cloiupyter',
 'billennium_regae',
 'cbroeckl_ramclustr',
 'jsgoecke_tesla',
 'robotology_ycm',
 'hasadna_anyway-newsflash-infographics']

In [32]:
assignments = pd.read_csv("net2prj.csv")

assignments.head()

,netID,WoC,GH
0,eabbott9,daducci_amico,https://github.com/daducci/AMICO
1,eabbott9,yangjasp_optimall,https://github.com/yangjasp/optimall
2,eabbott9,anybotics_kindr,https://github.com/ANYbotics/kindr
3,eabbott9,scikit-fuzzy_scikit-fuzzy,https://github.com/scikit-fuzzy/scikit-fuzzy
4,eabbott9,mhhennig_hs2,https://github.com/mhhennig/HS2


In [33]:
assignments.columns

Index(['netID', 'WoC', 'GH'], dtype='str')

In [34]:
assignments.head()

,netID,WoC,GH
0,eabbott9,daducci_amico,https://github.com/daducci/AMICO
1,eabbott9,yangjasp_optimall,https://github.com/yangjasp/optimall
2,eabbott9,anybotics_kindr,https://github.com/ANYbotics/kindr
3,eabbott9,scikit-fuzzy_scikit-fuzzy,https://github.com/scikit-fuzzy/scikit-fuzzy
4,eabbott9,mhhennig_hs2,https://github.com/mhhennig/HS2


In [35]:
my_projects = assignments[assignments["netID"] == "bxy539"]

my_projects

,netID,WoC,GH
760,bxy539,ewenharrison_finalfit,https://github.com/ewenharrison/finalfit
761,bxy539,lucidrains_lightweight-gan,https://github.com/lucidrains/lightweight-gan
762,bxy539,walaj_seqlib,https://github.com/walaj/SeqLib
763,bxy539,3dct_open_ia,https://github.com/3dct/open_iA
764,bxy539,clojupyter_clojupyter,https://github.com/clojupyter/clojupyter
765,bxy539,billennium_regae,https://github.com/billennium/ReGAE
766,bxy539,cbroeckl_ramclustr,https://github.com/cbroeckl/RAMClustR
767,bxy539,jsgoecke_tesla,https://github.com/jsgoecke/tesla
768,bxy539,robotology_ycm,https://github.com/robotology/ycm
769,bxy539,hasadna_anyway-newsflash-infographics,https://github.com/hasadna/anyway-newsflash-in...


In [36]:
projects = my_projects["WoC"].tolist()
projects

['ewenharrison_finalfit',
 'lucidrains_lightweight-gan',
 'walaj_seqlib',
 '3dct_open_ia',
 'clojupyter_clojupyter',
 'billennium_regae',
 'cbroeckl_ramclustr',
 'jsgoecke_tesla',
 'robotology_ycm',
 'hasadna_anyway-newsflash-infographics']

In [37]:
list_df_commits = []

for prj in projects:
    print("Retrieving:", prj)

    commits = woc.get_values('p2c', prj)

    temp_df = pd.DataFrame(commits, columns=['sha1'])
    temp_df['project'] = prj

    list_df_commits.append(temp_df)

df_commits = pd.concat(list_df_commits, ignore_index=True)

df_commits.head()

Retrieving: ewenharrison_finalfit
Retrieving: lucidrains_lightweight-gan
Retrieving: walaj_seqlib
Retrieving: 3dct_open_ia
Retrieving: clojupyter_clojupyter
Retrieving: billennium_regae
Retrieving: cbroeckl_ramclustr
Retrieving: jsgoecke_tesla
Retrieving: robotology_ycm
Retrieving: hasadna_anyway-newsflash-infographics


,sha1,project
0,00e7fb64c7796fc9e3fff1addfb4230861537755,ewenharrison_finalfit
1,00e9e6c2bc3795e07f7ea73cfbb031f8878035b8,ewenharrison_finalfit
2,029d8d94e7d076c5cec9818071708d4e9d6f56cf,ewenharrison_finalfit
3,02cbe88bf2b3934774e3473c3e22c34f6a81abf0,ewenharrison_finalfit
4,02de8a97ec53947e36c3662d457d67241ea6d6b3,ewenharrison_finalfit


In [42]:
chunks = [
    df_commits['sha1'][x:x+10]
    for x in range(0, len(df_commits), 10)
]

commit_data = []

print("Total commits:", len(df_commits))
print("Total chunks:", len(chunks))

Total commits: 17757
Total chunks: 1776


In [43]:
for chunk in tqdm(chunks):
    res, err = woc.get_values_many('commit.tch', chunk.to_list())

    res = {k: v[0] for k, v in res.items()}

    time.sleep(1)

    if err:
        print('Got Errors', err)

    for commit_sha, commit in res.items():
        commit_data.append({
            'commit': commit_sha,
            'tree': commit[0],
            'parent': list(commit[1]),
            'author': commit[2][0],
            'author_time': int(commit[2][1]),
            'author_tz': commit[2][2],
            'committer': commit[3][0],
            'committer_time': int(commit[3][1]),
            'committer_tz': commit[3][2],
            'message': commit[4],
        })

 10%|███████████████▊                                                                                                                                                 | 175/1776 [03:02<27:57,  1.05s/it]

Got Errors {'01518b0d37b796424b6542fe09321dc55bfacaf3': 'Key 01518b0d37b796424b6542fe09321dc55bfacaf3 not found in /da5_fast/All.sha1c/commit_1.tch'}


 12%|███████████████████▊                                                                                                                                             | 218/1776 [03:46<27:04,  1.04s/it]

Got Errors {'0e68ef8f65645bb360a189be36465094e913c81f': 'Key 0e68ef8f65645bb360a189be36465094e913c81f not found in /da5_fast/All.sha1c/commit_14.tch'}


 15%|████████████████████████▊                                                                                                                                        | 274/1776 [04:45<26:10,  1.05s/it]

...

Got Errors {'fc15599f0dfb52b6a2805ea542fcfc00da919f16': 'Key fc15599f0dfb52b6a2805ea542fcfc00da919f16 not found in /da5_fast/All.sha1c/commit_124.tch'}


 86%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍                      | 1525/1776 [26:36<04:19,  1.03s/it]

Got Errors {'fd79720a8b852807b9f3b292fd5fa1f731044cdb': 'Key fd79720a8b852807b9f3b292fd5fa1f731044cdb not found in /da5_fast/All.sha1c/commit_125.tch'}


 86%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌                      | 1527/1776 [26:43<04:21,  1.05s/it]


ReadTimeout: 

In [44]:
print("Commit details successfully retrieved:", len(commit_data))
print("Total commits:", len(df_commits))
print("Remaining:", len(df_commits) - len(commit_data))

Commit details successfully retrieved: 15232
Total commits: 17757
Remaining: 2525


In [45]:
# SHA1s that were already successfully retrieved
retrieved_sha1s = {item['commit'] for item in commit_data}

# Find commits that are still missing
missing_df = df_commits[
    ~df_commits['sha1'].isin(retrieved_sha1s)
].copy()

print("Already retrieved:", len(retrieved_sha1s))
print("Missing commits:", len(missing_df))

missing_df.head()

Already retrieved: 15232
Missing commits: 2525


,sha1,project
1740,01518b0d37b796424b6542fe09321dc55bfacaf3,3dct_open_ia
2173,0e68ef8f65645bb360a189be36465094e913c81f,3dct_open_ia
2738,1e914e15f402e80428f6eb93708861d3c6e5dab0,3dct_open_ia
2862,21ee999ce45faa9adbaeef1f22e0886253781003,3dct_open_ia
3326,2e672b911bdb446e48a64532c227d6c8bb5f40be,3dct_open_ia


In [46]:
missing_chunks = [
    missing_df['sha1'][x:x+10]
    for x in range(0, len(missing_df), 10)
]

print("Missing commits:", len(missing_df))
print("Missing chunks:", len(missing_chunks))

Missing commits: 2525
Missing chunks: 253


In [47]:
for chunk in tqdm(missing_chunks):
    try:
        res, err = woc.get_values_many('commit.tch', chunk.to_list())
        res = {k: v[0] for k, v in res.items()}

        if err:
            print('Got Errors:', err)

        for commit_sha, commit in res.items():
            commit_data.append({
                'commit': commit_sha,
                'tree': commit[0],
                'parent': list(commit[1]),
                'author': commit[2][0],
                'author_time': int(commit[2][1]),
                'author_tz': commit[2][2],
                'committer': commit[3][0],
                'committer_time': int(commit[3][1]),
                'committer_tz': commit[3][2],
                'message': commit[4],
            })

        time.sleep(1)

    except Exception as e:
        print("Error on chunk:", e)
        time.sleep(5)

  0%|                                                                                                                                                                            | 0/253 [00:00<?, ?it/s]

Got Errors: {'01518b0d37b796424b6542fe09321dc55bfacaf3': 'Key 01518b0d37b796424b6542fe09321dc55bfacaf3 not found in /da5_fast/All.sha1c/commit_1.tch', '0e68ef8f65645bb360a189be36465094e913c81f': 'Key 0e68ef8f65645bb360a189be36465094e913c81f not found in /da5_fast/All.sha1c/commit_14.tch', '1e914e15f402e80428f6eb93708861d3c6e5dab0': 'Key 1e914e15f402e80428f6eb93708861d3c6e5dab0 not found in /da5_fast/All.sha1c/commit_30.tch', '21ee999ce45faa9adbaeef1f22e0886253781003': 'Key 21ee999ce45faa9adbaeef1f22e0886253781003 not found in /da5_fast/All.sha1c/commit_33.tch', '2e672b911bdb446e48a64532c227d6c8bb5f40be': 'Key 2e672b911bdb446e48a64532c227d6c8bb5f40be not found in /da5_fast/All.sha1c/commit_46.tch', '41028dbfe00c18055bb1c1d21768897e96dd235c': 'Key 41028dbfe00c18055bb1c1d21768897e96dd235c not found in /da5_fast/All.sha1c/commit_65.tch', '411f8fc340c340af2efa3ec74d9e168b3c40aa19': 'Key 411f8fc340c340af2efa3ec74d9e168b3c40aa19 not found in /da5_fast/All.sha1c/commit_65.tch', '4a44af278f1916

  0%|▋                                                                                                                                                                   | 1/253 [00:01<04:17,  1.02s/it]

Got Errors: {'9f7b2c6095538781584c5ee6ae9c6817def900c2': 'Key 9f7b2c6095538781584c5ee6ae9c6817def900c2 not found in /da5_fast/All.sha1c/commit_31.tch', 'a3d6d354e8cff7251a3babc13a12337f329d2c0b': 'Key a3d6d354e8cff7251a3babc13a12337f329d2c0b not found in /da5_fast/All.sha1c/commit_35.tch', 'e7292aa5a9d40202141531a28cafcffb08593767': 'Key e7292aa5a9d40202141531a28cafcffb08593767 not found in /da5_fast/All.sha1c/commit_103.tch', '9f307bc91ba008d3078ed1fe9a40da880ba33dec': 'Key 9f307bc91ba008d3078ed1fe9a40da880ba33dec not found in /da5_fast/All.sha1c/commit_31.tch', 'b36c23f47aa8864617af6847e9056f94825b9c1d': 'Key b36c23f47aa8864617af6847e9056f94825b9c1d not found in /da5_fast/All.sha1c/commit_51.tch', '108107df8109060c9a3870a6433da56d9131da16': 'Key 108107df8109060c9a3870a6433da56d9131da16 not found in /da5_fast/All.sha1c/commit_16.tch', '152edb36c3361815bc88f1c2dfd0c59fe2486cba': 'Key 152edb36c3361815bc88f1c2dfd0c59fe2486cba not found in /da5_fast/All.sha1c/commit_21.tch', '1874cd5d06c1

  1%|█▎                                                                                                                                                                  | 2/253 [00:02<04:16,  1.02s/it]

Got Errors: {'2c08d298ff986c182c71a9addfee1ace0ade9e69': 'Key 2c08d298ff986c182c71a9addfee1ace0ade9e69 not found in /da5_fast/All.sha1c/commit_44.tch', '2e68b07931bc467cc8514a47d6687d206894d943': 'Key 2e68b07931bc467cc8514a47d6687d206894d943 not found in /da5_fast/All.sha1c/commit_46.tch', '3e8e08ba89cc78bf0d7c547829a4652dc23a15ba': 'Key 3e8e08ba89cc78bf0d7c547829a4652dc23a15ba not found in /da5_fast/All.sha1c/commit_62.tch', '68757ca73dddb1be9747eb1b618c90a6f042c203': 'Key 68757ca73dddb1be9747eb1b618c90a6f042c203 not found in /da5_fast/All.sha1c/commit_104.tch', '7e9a0285fb2ff1d17bd99c8c0704a13dab496dca': 'Key 7e9a0285fb2ff1d17bd99c8c0704a13dab496dca not found in /da5_fast/All.sha1c/commit_126.tch', '8a9594300d5a401d1f388209d7c81a62f24bd3f1': 'Key 8a9594300d5a401d1f388209d7c81a62f24bd3f1 not found in /da5_fast/All.sha1c/commit_10.tch', '8e5d3561107933aa1e11719bcdfe89a2f0fb5c2b': 'Key 8e5d3561107933aa1e11719bcdfe89a2f0fb5c2b not found in /da5_fast/All.sha1c/commit_14.tch', '96498c5aec8

  1%|█▉                                                                                                                                                                  | 3/253 [00:03<04:14,  1.02s/it]

Got Errors: {'c078daa199c6553d280175c8801d353ec568694f': 'Key c078daa199c6553d280175c8801d353ec568694f not found in /da5_fast/All.sha1c/commit_64.tch', 'ca9724756a109ec4b21a897c5510b93e1ae39298': 'Key ca9724756a109ec4b21a897c5510b93e1ae39298 not found in /da5_fast/All.sha1c/commit_74.tch', 'd73766d8d282f4375bb10365d4c8b668a61456a2': 'Key d73766d8d282f4375bb10365d4c8b668a61456a2 not found in /da5_fast/All.sha1c/commit_87.tch', 'daf25d17f2a856ad07d7f252e7adb568e25a79af': 'Key daf25d17f2a856ad07d7f252e7adb568e25a79af not found in /da5_fast/All.sha1c/commit_90.tch', 'f1b0686f8c112aa8bd85e929494cd9bf1c12e14c': 'Key f1b0686f8c112aa8bd85e929494cd9bf1c12e14c not found in /da5_fast/All.sha1c/commit_113.tch', 'f46664d53eced7ffb9529c27b854030b7d7221c2': 'Key f46664d53eced7ffb9529c27b854030b7d7221c2 not found in /da5_fast/All.sha1c/commit_116.tch', 'fc15599f0dfb52b6a2805ea542fcfc00da919f16': 'Key fc15599f0dfb52b6a2805ea542fcfc00da919f16 not found in /da5_fast/All.sha1c/commit_124.tch', 'fd79720a8b

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 253/253 [04:25<00:00,  1.05s/it]


In [48]:
retrieved_sha1s = {item['commit'] for item in commit_data}

still_missing = df_commits[
    ~df_commits['sha1'].isin(retrieved_sha1s)
].copy()

print("Total commits:", len(df_commits))
print("Successfully retrieved:", len(retrieved_sha1s))
print("Still missing:", len(still_missing))

Total commits: 17757
Successfully retrieved: 17719
Still missing: 38


In [49]:
for sha in tqdm(still_missing['sha1']):
    try:
        res, err = woc.get_values_many('commit.tch', [sha])

        if res:
            res = {k: v[0] for k, v in res.items()}

            for commit_sha, commit in res.items():
                commit_data.append({
                    'commit': commit_sha,
                    'tree': commit[0],
                    'parent': list(commit[1]),
                    'author': commit[2][0],
                    'author_time': int(commit[2][1]),
                    'author_tz': commit[2][2],
                    'committer': commit[3][0],
                    'committer_time': int(commit[3][1]),
                    'committer_tz': commit[3][2],
                    'message': commit[4],
                })

        time.sleep(1)

    except Exception as e:
        print("Could not retrieve:", sha, e)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 38/38 [00:38<00:00,  1.02s/it]


In [50]:
retrieved_sha1s = {item['commit'] for item in commit_data}

still_missing = df_commits[
    ~df_commits['sha1'].isin(retrieved_sha1s)
].copy()

print("Successfully retrieved:", len(retrieved_sha1s))
print("Still missing:", len(still_missing))

Successfully retrieved: 17719
Still missing: 38


In [51]:
# Convert retrieved commit details to a DataFrame
details_df = pd.DataFrame(commit_data)

# Keep only the columns required for the assignment
details_df = details_df[
    ['commit', 'author', 'author_time', 'message']
].copy()

# Rename columns to match assignment requirements
details_df = details_df.rename(columns={
    'commit': 'commit_sha1',
    'author_time': 'time',
    'message': 'commit message'
})

details_df.head()

,commit_sha1,author,time,commit message
0,00e7fb64c7796fc9e3fff1addfb4230861537755,ewenharrison <mail@ewenharrison.com>,1556099922,Fix export vignette\n
1,00e9e6c2bc3795e07f7ea73cfbb031f8878035b8,ewenharrison <mail@ewenharrison.com>,1542882428,Merge branch 'master' of github.com:ewenharris...
2,029d8d94e7d076c5cec9818071708d4e9d6f56cf,ewenharrison <mail@ewenharrison.com>,1538596968,Merge branch 'master' of github.com:ewenharris...
3,02cbe88bf2b3934774e3473c3e22c34f6a81abf0,ewenharrison <mail@ewenharrison.com>,1549052653,Merge branch 'master' of github.com:ewenharris...
4,02de8a97ec53947e36c3662d457d67241ea6d6b3,ewenharrison <mail@ewenharrison.com>,1536658512,New vignettes\n


In [52]:
details_df = details_df.merge(
    df_commits[['sha1', 'project']],
    left_on='commit_sha1',
    right_on='sha1',
    how='left'
)

details_df = details_df.rename(columns={'project': 'project_wocid'})

details_df = details_df[
    ['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']
]

details_df.head()

,project_wocid,commit_sha1,author,time,commit message
0,ewenharrison_finalfit,00e7fb64c7796fc9e3fff1addfb4230861537755,ewenharrison <mail@ewenharrison.com>,1556099922,Fix export vignette\n
1,ewenharrison_finalfit,00e9e6c2bc3795e07f7ea73cfbb031f8878035b8,ewenharrison <mail@ewenharrison.com>,1542882428,Merge branch 'master' of github.com:ewenharris...
2,ewenharrison_finalfit,029d8d94e7d076c5cec9818071708d4e9d6f56cf,ewenharrison <mail@ewenharrison.com>,1538596968,Merge branch 'master' of github.com:ewenharris...
3,ewenharrison_finalfit,02cbe88bf2b3934774e3473c3e22c34f6a81abf0,ewenharrison <mail@ewenharrison.com>,1549052653,Merge branch 'master' of github.com:ewenharris...
4,ewenharrison_finalfit,02de8a97ec53947e36c3662d457d67241ea6d6b3,ewenharrison <mail@ewenharrison.com>,1536658512,New vignettes\n


In [55]:
final_df = details_df.merge(
    df_commits[['sha1', 'project']],
    left_on='commit_sha1',
    right_on='sha1',
    how='left'
)

final_df = final_df.rename(columns={'project': 'project_wocid'})

final_df = final_df[
    ['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']
]

final_df.head()

,project_wocid,project_wocid,commit_sha1,author,time,commit message
0,ewenharrison_finalfit,ewenharrison_finalfit,00e7fb64c7796fc9e3fff1addfb4230861537755,ewenharrison <mail@ewenharrison.com>,1556099922,Fix export vignette\n
1,ewenharrison_finalfit,ewenharrison_finalfit,00e9e6c2bc3795e07f7ea73cfbb031f8878035b8,ewenharrison <mail@ewenharrison.com>,1542882428,Merge branch 'master' of github.com:ewenharris...
2,ewenharrison_finalfit,ewenharrison_finalfit,029d8d94e7d076c5cec9818071708d4e9d6f56cf,ewenharrison <mail@ewenharrison.com>,1538596968,Merge branch 'master' of github.com:ewenharris...
3,ewenharrison_finalfit,ewenharrison_finalfit,02cbe88bf2b3934774e3473c3e22c34f6a81abf0,ewenharrison <mail@ewenharrison.com>,1549052653,Merge branch 'master' of github.com:ewenharris...
4,ewenharrison_finalfit,ewenharrison_finalfit,02de8a97ec53947e36c3662d457d67241ea6d6b3,ewenharrison <mail@ewenharrison.com>,1536658512,New vignettes\n


In [56]:
# Remove duplicate columns
final_df = final_df.loc[:, ~final_df.columns.duplicated()]

final_df.head()

,project_wocid,commit_sha1,author,time,commit message
0,ewenharrison_finalfit,00e7fb64c7796fc9e3fff1addfb4230861537755,ewenharrison <mail@ewenharrison.com>,1556099922,Fix export vignette\n
1,ewenharrison_finalfit,00e9e6c2bc3795e07f7ea73cfbb031f8878035b8,ewenharrison <mail@ewenharrison.com>,1542882428,Merge branch 'master' of github.com:ewenharris...
2,ewenharrison_finalfit,029d8d94e7d076c5cec9818071708d4e9d6f56cf,ewenharrison <mail@ewenharrison.com>,1538596968,Merge branch 'master' of github.com:ewenharris...
3,ewenharrison_finalfit,02cbe88bf2b3934774e3473c3e22c34f6a81abf0,ewenharrison <mail@ewenharrison.com>,1549052653,Merge branch 'master' of github.com:ewenharris...
4,ewenharrison_finalfit,02de8a97ec53947e36c3662d457d67241ea6d6b3,ewenharrison <mail@ewenharrison.com>,1536658512,New vignettes\n


In [57]:
# Remove duplicate columns
final_df = final_df.loc[:, ~final_df.columns.duplicated()]

final_df.head()

,project_wocid,commit_sha1,author,time,commit message
0,ewenharrison_finalfit,00e7fb64c7796fc9e3fff1addfb4230861537755,ewenharrison <mail@ewenharrison.com>,1556099922,Fix export vignette\n
1,ewenharrison_finalfit,00e9e6c2bc3795e07f7ea73cfbb031f8878035b8,ewenharrison <mail@ewenharrison.com>,1542882428,Merge branch 'master' of github.com:ewenharris...
2,ewenharrison_finalfit,029d8d94e7d076c5cec9818071708d4e9d6f56cf,ewenharrison <mail@ewenharrison.com>,1538596968,Merge branch 'master' of github.com:ewenharris...
3,ewenharrison_finalfit,02cbe88bf2b3934774e3473c3e22c34f6a81abf0,ewenharrison <mail@ewenharrison.com>,1549052653,Merge branch 'master' of github.com:ewenharris...
4,ewenharrison_finalfit,02de8a97ec53947e36c3662d457d67241ea6d6b3,ewenharrison <mail@ewenharrison.com>,1536658512,New vignettes\n


In [58]:
print("Rows:", len(final_df))
print("Missing project IDs:", final_df['project_wocid'].isna().sum())

print("\nCommits per project:")
print(final_df['project_wocid'].value_counts())

Rows: 17719
Missing project IDs: 0

Commits per project:
project_wocid
3dct_open_ia                             9095
hasadna_anyway-newsflash-infographics    2495
robotology_ycm                           2049
walaj_seqlib                              893
cbroeckl_ramclustr                        805
clojupyter_clojupyter                     753
billennium_regae                          454
ewenharrison_finalfit                     436
lucidrains_lightweight-gan                373
jsgoecke_tesla                            366
Name: count, dtype: int64


In [60]:
check_df = pd.read_csv("bxy539_project_summary.csv", sep=";")

print(check_df.shape)
print(check_df.columns.tolist())
check_df.head()

(17719, 5)
['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']


,project_wocid,commit_sha1,author,time,commit message
0,ewenharrison_finalfit,00e7fb64c7796fc9e3fff1addfb4230861537755,ewenharrison <mail@ewenharrison.com>,1556099922,Fix export vignette\n
1,ewenharrison_finalfit,00e9e6c2bc3795e07f7ea73cfbb031f8878035b8,ewenharrison <mail@ewenharrison.com>,1542882428,Merge branch 'master' of github.com:ewenharris...
2,ewenharrison_finalfit,029d8d94e7d076c5cec9818071708d4e9d6f56cf,ewenharrison <mail@ewenharrison.com>,1538596968,Merge branch 'master' of github.com:ewenharris...
3,ewenharrison_finalfit,02cbe88bf2b3934774e3473c3e22c34f6a81abf0,ewenharrison <mail@ewenharrison.com>,1549052653,Merge branch 'master' of github.com:ewenharris...
4,ewenharrison_finalfit,02de8a97ec53947e36c3662d457d67241ea6d6b3,ewenharrison <mail@ewenharrison.com>,1536658512,New vignettes\n


In [59]:
final_df.to_csv(
    "bxy539_project_summary.csv",
    sep=";",
    index=False
)

print("CSV created successfully!")

CSV created successfully!


In [62]:
woc_stats = final_df.groupby('project_wocid').agg(
    number_of_commits=('commit_sha1', 'count'),
    number_of_authors=('author', 'nunique'),
    max_time=('time', 'max'),
    min_time=('time', 'min')
)

woc_stats

,number_of_commits,number_of_authors,max_time,min_time
project_wocid,,,,
3dct_open_ia,9095,40,1774994202,1464945771
billennium_regae,454,7,1731935108,1634055684
cbroeckl_ramclustr,805,18,1775245947,1382511847
clojupyter_clojupyter,753,48,1742327393,1399620581
ewenharrison_finalfit,436,11,1761574972,1526162607
hasadna_anyway-newsflash-infographics,2495,97,1753562244,1576090079
jsgoecke_tesla,366,32,1733042460,1460485310
lucidrains_lightweight-gan,373,17,1736647889,1605115894
robotology_ycm,2049,47,1752750785,1146563034


In [63]:
</> Markdown
## WoC Project Statistics

| Project | Number of Commits | Number of Authors | Max Time | Min Time |
|---|---:|---:|---:|---:|
| 3dct_open_ia | 9095 | 40 | 1774994202 | 1464945771 |
| billennium_regae | 454 | 7 | 1731935108 | 1634055684 |
| cbroeckl_ramclustr | 805 | 18 | 1775245947 | 1382511847 |
| clojupyter_clojupyter | 753 | 48 | 1742327393 | 1399620581 |
| ewenharrison_finalfit | 436 | 11 | 1761574972 | 1526162607 |
| hasadna_anyway-newsflash-infographics | 2495 | 97 | 1753562244 | 1576090079 |
| jsgoecke_tesla | 366 | 32 | 1733042460 | 1460485310 |
| lucidrains_lightweight-gan | 373 | 17 | 1736647889 | 1605115894 |
| robotology_ycm | 2049 | 47 | 1752750785 | 1146563034 |
| walaj_seqlib | 893 | 24 | 1758045612 | 1431016848 |

SyntaxError: invalid syntax (3654152709.py, line 1)

In [66]:
import requests

github_repos = {
    "ewenharrison_finalfit": "ewenharrison/finalfit",
    "lucidrains_lightweight-gan": "lucidrains/lightweight-gan",
    "walaj_seqlib": "walaj/seqlib",
    "3dct_open_ia": "3dct/open_ia",
    "clojupyter_clojupyter": "clojupyter/clojupyter",
    "billennium_regae": "billennium/regae",
    "cbroeckl_ramclustr": "cbroeckl/ramclustr",
    "jsgoecke_tesla": "jsgoecke/tesla",
    "robotology_ycm": "robotology/ycm",
    "hasadna_anyway-newsflash-infographics": "hasadna/anyway-newsflash-infographics"
}

In [67]:
github_data = []

for project, repo in github_repos.items():
    url = f"https://api.github.com/repos/{repo}"
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()

        github_data.append({
            "project": project,
            "stars": data["stargazers_count"],
            "forks": data["forks_count"],
            "last_commit_date": data["pushed_at"]
        })

        print("Retrieved:", project)
    else:
        print("ERROR:", project, response.status_code)

Retrieved: ewenharrison_finalfit
Retrieved: lucidrains_lightweight-gan
Retrieved: walaj_seqlib
Retrieved: 3dct_open_ia
Retrieved: clojupyter_clojupyter
Retrieved: billennium_regae
Retrieved: cbroeckl_ramclustr
Retrieved: jsgoecke_tesla
Retrieved: robotology_ycm
Retrieved: hasadna_anyway-newsflash-infographics


In [68]:
github_df = pd.DataFrame(github_data)

github_df

,project,stars,forks,last_commit_date
0,ewenharrison_finalfit,276,27,2025-10-27T14:22:52Z
1,lucidrains_lightweight-gan,1686,218,2025-01-12T02:11:34Z
2,walaj_seqlib,139,38,2026-04-27T17:17:42Z
3,3dct_open_ia,48,20,2026-08-28T05:09:51Z
4,clojupyter_clojupyter,868,96,2025-03-18T08:36:00Z
5,billennium_regae,3,1,2024-11-18T13:05:10Z
6,cbroeckl_ramclustr,13,18,2026-04-03T19:52:30Z
7,jsgoecke_tesla,327,75,2024-12-01T08:41:00Z
8,robotology_ycm,63,24,2026-07-24T08:34:38Z
9,hasadna_anyway-newsflash-infographics,13,22,2025-07-10T03:16:55Z


In [69]:
final_df.to_csv(
    "bpatel_project_summary.csv",
    sep=";",
    index=False
)

print("CSV file created successfully!")

CSV file created successfully!


In [70]:
check_df = pd.read_csv("bpatel_project_summary.csv", sep=";")

print(check_df.shape)
print(check_df.columns.tolist())
check_df.head()

(17719, 5)
['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']


,project_wocid,commit_sha1,author,time,commit message
0,ewenharrison_finalfit,00e7fb64c7796fc9e3fff1addfb4230861537755,ewenharrison <mail@ewenharrison.com>,1556099922,Fix export vignette\n
1,ewenharrison_finalfit,00e9e6c2bc3795e07f7ea73cfbb031f8878035b8,ewenharrison <mail@ewenharrison.com>,1542882428,Merge branch 'master' of github.com:ewenharris...
2,ewenharrison_finalfit,029d8d94e7d076c5cec9818071708d4e9d6f56cf,ewenharrison <mail@ewenharrison.com>,1538596968,Merge branch 'master' of github.com:ewenharris...
3,ewenharrison_finalfit,02cbe88bf2b3934774e3473c3e22c34f6a81abf0,ewenharrison <mail@ewenharrison.com>,1549052653,Merge branch 'master' of github.com:ewenharris...
4,ewenharrison_finalfit,02de8a97ec53947e36c3662d457d67241ea6d6b3,ewenharrison <mail@ewenharrison.com>,1536658512,New vignettes\n


In [71]:
df = pd.read_csv("example_project_summary.csv", sep=";")
df.columns = ['project','commit','author','time','message']
df['Month'] = pd.to_datetime(df['time'], unit='s').dt.strftime('%Y-%m')
df.head()

ParserError: Error tokenizing data. C error: Expected 1 fields in line 258, saw 2


In [40]:
prj='mmorise_world'
df1 = df[df['project']==prj]
df2 = df1.groupby('Month').size().reset_index(name='Commits')
monthly_counts = df2
monthly_counts['time'] = pd.to_datetime(monthly_counts['Month'])
full_date_range = pd.DataFrame({'time': pd.date_range(start=monthly_counts['time'].min(), end=monthly_counts['time'].max(), freq='MS')})
monthly_counts = pd.merge(full_date_range, monthly_counts, on='time', how='left').fillna(0)
monthly_counts['Month'] = monthly_counts['time'].dt.strftime('%Y-%m')


NameError: name 'df' is not defined

In [41]:
# Plot
import matplotlib.ticker as ticker

plt.figure(figsize=(10,5))
plt.plot(monthly_counts["Month"], monthly_counts["Commits"], marker="o", linestyle="-")
plt.xlabel("Time (YYYY-MM)")
plt.ylabel("Number of Commits")
plt.title("Commit Activity: "+prj)
plt.grid(True)
plt.tight_layout()
ax =  plt.gca()
for label in ax.xaxis.get_ticklabels()[::2]:
    label.set_visible(False)
plt.xticks(rotation=45, fontsize=7)
plt.savefig("your-net-id-here_timeseries_"+prj+".png", dpi=600)
plt.show()
plt.close()

NameError: name 'monthly_counts' is not defined

<Figure size 1000x500 with 0 Axes>

# Identify the Longest Inactivity Gap

2022-02 to 2023-04

Add all that info to netid_project_stats.csv